Let's do some performance benchmarking comparing to Orbit

TODO:
Redefine name:
- no more beta; everything call 
- regressor matrix (reg_mat)
- regressor
- coef (no beta)
- combine trend and intercept
- a module to construct loc and scale for different type of regressors

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import logging

# import os
# os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=true --xla_cpu_multi_thread_eigen_num_threads=4"
import numpyro
numpyro.set_host_device_count(4)
import matplotlib.pyplot as plt

from wunku.models.dlt import (
    run_dlt_model, 
    generate_in_sample_forecast, 
)
from wunku.hyper_tune import (
    generate_forecast_span_samples,
    compute_log_likelihood,
    compute_wbic,
    run_dlt_model_and_compute_wbic,
    hyper_tuning_dlt_with_wbic,
)
from wunku.utils import (
    summarize_posteriors,
    make_fourier_series,
)

In [ ]:
# Get or create logger
logger = logging.getLogger("wunku")

# Set logger level
logger.setLevel(logging.DEBUG)

# Add a StreamHandler if no handlers are attached
if not logger.hasHandlers():
    handler = logging.StreamHandler()
    handler.setLevel(logging.DEBUG)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)

    logger.addHandler(handler)

# Example usage
logger.debug("This is a DEBUG message")
logger.info("This is an INFO message")

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
model_tag = "simple"
df = pd.read_csv(f"../resource/{model_tag}/df.csv")
saturation_df = pd.read_csv(f"../resource/{model_tag}/saturation_df.csv").set_index("regressor")
response = df["sales"].values
# response_norm = response.mean()
response_norm = 1.
y = np.log(response / response_norm)

resource_cols = [
    "promo", "radio", "search", "social", "tv"
]

In [ ]:
x_glb_trend = np.arange(len(y)) / 365.25
# x_annual_seas = make_fourier_series(np.arange(len(y)), period=365.25, order=3)
# x_weekly_seas = make_fourier_series(np.arange(len(y)), period=7, order=2)
# x_seas = np.concatenate((x_annual_seas, x_weekly_seas), axis=1)
# print(x_seas.shape)
x_resource = df[resource_cols].values
sat_arr = saturation_df.loc[resource_cols, "saturation"].values
x_resource = np.log1p(x_resource / sat_arr)

In [ ]:
# from our best model last time
lev_sm=0.0266
# lev_sm = 0.00245
slp_sm=0.0016
# theta=0.94900
theta=0.00

In [ ]:
data_vars = {
    "covariates": (["time", "regressor"], x_resource),
    "coef_loc": (["regressor"], np.zeros(len(resource_cols))),
    "coef_scale": (["regressor"], np.full(len(resource_cols), 0.1)),
}
coords = {
    "time": np.arange(len(y)),
    "regressor": resource_cols,
}
regression_scheme = xr.Dataset(
    data_vars=data_vars,
    coords=coords,
)
regression_scheme

In [ ]:
posteriors = run_dlt_model(
    y=y,
    # retire this as well later
    x_glb_trend=x_glb_trend,
    lev_sm=lev_sm,
    slp_sm=slp_sm,
    theta=theta,
    regression_scheme=regression_scheme,
)

In [ ]:
import arviz as az
summary_df = az.summary(
    posteriors,
    # var_names=["alpha_glb_trend", "beta_glb_trend"],
    var_names=["alpha_glb_trend", "beta_glb_trend", "beta_ext", "sigma"],
)
summary_df

In [ ]:
coefs_df = pd.read_csv(f"../resource/{model_tag}/coefs_df.csv", index_col="regressor")
coefs_df

In [ ]:
beta_ext_names = [f"beta_ext[{x}]" for x in resource_cols]
coef_p50 = summary_df.loc[beta_ext_names, "mean"].values
coef_true = coefs_df.loc[resource_cols, "coef"].values
mse_err = np.mean(
    np.abs(coef_true - coef_p50)
)

smape_err = np.mean(
    2 * np.abs(coef_true - coef_p50) / (np.abs(coef_true) + np.abs(coef_p50))
)

print(f"Coef SMAPE (%): {smape_err:.3%}")
print(f"MSE: {mse_err:.4f}")